# 🎲 SDXL 随机生成（随机种子）

使用本地模型 `stable-diffusion-xl-base-1.0`（SDXL，2.6B 参数，原生 1024×1024）。

**随机种子原理**：SD 生成图的起点是一张噪声图，噪声由种子（seed）决定——
- 同一 seed + 同一 prompt + 同一参数 → 永远生成同一张图（可复现）
- seed 随机 → 每次都是全新构图

**8GB 显存适配**：fp16 半精度 + `enable_model_cpu_offload()`（组件按需进出 GPU），实测显存峰值约 5.2 GB，无需量化。

请选择内核：**transformers (15-transformers-basics)**

In [ ]:
# 1. 环境检查
import torch

print(f'torch: {torch.__version__} | CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# 2. 加载 SDXL 管线（只加载一次，后续 cell 复用；加载约 10~60 秒）
import torch
from diffusers import StableDiffusionXLPipeline

MODEL_DIR = r'S:\PythonAILearningCourse\04-ai-painting-sdxl\stable-diffusion-xl-base-1.0'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,   # 半精度
    variant='fp16',        # 权重文件名带 .fp16 后缀，必须指定
)
pipe.enable_model_cpu_offload()   # 关键：8GB 显存跑 6.9GB 权重，组件按需进出 GPU
# 注：SDXL 的 VAE 在 fp16 下会溢出（黑图），pipeline 会依据 vae config 的
#     force_upcast=true 在解码时自动转 fp32，无需手动处理

print('✅ SDXL 管线加载完成')

In [ ]:
# 3. 随机种子生成：每次运行本 cell 都会得到一张全新的图（1024x1024 约 30~60 秒）
import random

prompt = 'a majestic lion on a rock at sunset, golden light, photorealistic, highly detailed'
negative_prompt = 'lowres, bad anatomy, blurry, watermark, text'

# ---- 随机种子：生成范围内的随机整数 ----
seed = torch.randint(0, 2**31 - 1, (1,)).item()   # 随机抽一个种子
print(f'本次随机 seed = {seed}')   # 记下它！用第 4 个 cell 可以复现同一张图

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,       # SDXL 1024x1024 推荐步数 25~40
    guidance_scale=5.0,           # SDXL 常用 4~7（比 SD1.5 的 7.5 低）
    height=1024, width=1024,      # SDXL 原生分辨率
    generator=torch.Generator('cuda').manual_seed(seed),  # 种子决定初始噪声
).images[0]

# 文件名带上 seed，方便以后找到喜欢的图
out = rf'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\sdxl_seed{seed}.png'
image.save(out)
print(f'已保存: {out}')
image

In [ ]:
# 4. 可复现：把 SEED 换成第 3 个 cell 打印过的数字，同样参数会得到一模一样的图
SEED = 123456789          # ← 改成你记录的 seed

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=5.0,
    height=1024, width=1024,
    generator=torch.Generator('cuda').manual_seed(SEED),
).images[0]
image